# Enterprise AI Audio/Video Analyzer

This project is a complete AI pipeline designed to process any audio or video file (or Google Drive link). The primary goal of this system is to analyze long videos, generate comprehensive summaries, and answer questions based on their content.

**Key Features:**
* **Whisper AI (Large-v3):** Automatically transcribes and translates audio from any language directly into English.
* **FAISS Vector Database:** Converts text chunks into vector embeddings for fast and accurate context retrieval.
* **Llama 3.1 (via ChatGroq):** Utilizes a Map-Reduce strategy to generate detailed summaries of long videos and powers the Q&A chatbot.
* **Gradio UI:** An interactive user interface for uploading files and chatting with the content.

**Important Note:** For optimal transcription accuracy, please avoid using AI-generated voices. Original, human voices yield the best results.

## 1. Core Dependencies Installation

In this section, we install the fundamental libraries required for the project. This includes LangChain (for LLM orchestration), FAISS (for the Vector Database), Gradio (for the User Interface), and Whisper/Sentence-Transformers (for Audio Processing and Embeddings).

In [1]:
# Install all core dependencies for local processing, AI embeddings, and UI
!pip install -q --upgrade langchain langchain-core langchain-community langchain-groq langchain-huggingface langchain-text-splitters faiss-cpu gradio sentence-transformers pydub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.7/588.7 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.3/234.3 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/4

## 2. Environment Setup (API Keys)

To run the Llama 3.1 model quickly and efficiently, we are using the Groq API. Here, we fetch the `GROQ_API_KEY` from Google Colab's "Secrets" and set it as an environment variable to ensure the code runs securely.

In [2]:
import os
from google.colab import userdata

try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    print("[Success] Environment Setup and API Key configured successfully.")
except userdata.SecretNotFoundError:
    print("[Error] 'GROQ_API_KEY' not found in Colab Secrets. Please check your configuration.")

[Success] Environment Setup and API Key configured successfully.


## 3. Media Processing & Groq Whisper API (Chunking & Transcription)

In this section, we process the media files and use Groq's lightning-fast `whisper-large-v3` cloud API for transcription. This function:
1. Handles both public Google Drive links and locally uploaded files.
2. Uses `pydub` to divide the audio into 10-minute chunks to effectively bypass API file size limits.
3. Detects the original language and automatically translates and transcribes the audio directly into English text via the Groq API.

In [3]:
import os
import gdown
import re
import time
from groq import Groq
from pydub import AudioSegment

def download_from_drive(url):
    try:
        file_id_match = re.search(r'/d/([a-zA-Z0-9-_]+)', url)
        if not file_id_match:
            file_id_match = re.search(r'id=([a-zA-Z0-9-_]+)', url)

        if file_id_match:
            file_id = file_id_match.group(1)
            download_url = f'https://drive.google.com/uc?id={file_id}'
            output_path = f'drive_media_{file_id}.mp4'

            print(f"Downloading from Google Drive... (ID: {file_id})")
            gdown.download(download_url, output_path, quiet=False)
            return output_path
        else:
            return None
    except Exception as e:
        print(f"Download Error: {str(e)}")
        return None

def process_media(file_path=None, drive_url=None):
    try:
        target_file = None

        if file_path:
            target_file = file_path
            print(f"Local file detected. Starting processing...")
        elif drive_url:
            target_file = download_from_drive(drive_url)
            if not target_file:
                return "Error: Invalid Google Drive link."
            print(f"Drive file downloaded. Starting processing...")

        if not target_file:
            return "Error: Please upload a file OR provide a valid Google Drive link."

        print("Preparing media file for API limits (Chunking)...")
        # File ko AudioSegment me load karain
        audio = AudioSegment.from_file(target_file)

        # 10 minutes ke chunks banayen (10 minutes * 60 seconds * 1000 milliseconds)
        chunk_length_ms = 10 * 60 * 1000
        chunks = [audio[i:i+chunk_length_ms] for i in range(0, len(audio), chunk_length_ms)]

        client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
        full_transcript = ""

        print(f"File divided into {len(chunks)} chunks. Transcribing via Groq API...")

        for index, chunk in enumerate(chunks):
            print(f"Processing chunk {index + 1} of {len(chunks)}...")
            chunk_name = f"temp_chunk_{index}.mp3"

            # Chunk ko export karain
            chunk.export(chunk_name, format="mp3")

            # Groq API per bhejen
            with open(chunk_name, "rb") as file:
                translation = client.audio.translations.create(
                  file=(chunk_name, file.read()),
                  model="whisper-large-v3",
                )

            full_transcript += translation.text + " "

            # Process hone ke baad temporary chunk delete ker dain
            os.remove(chunk_name)

            # API Rate limit se bachne ke liye 3 second ka gap
            time.sleep(3)

        print("[Success] Full transcription and translation complete via Groq.")

        # Original downloaded file delete ker dain
        if drive_url and os.path.exists(target_file):
            os.remove(target_file)

        return full_transcript.strip()

    except Exception as e:
        return f"Error processing media: {str(e)}"

print("[Success] Media Processor (Chunking & Groq API) loaded.")

[Success] Media Processor (Chunking & Groq API) loaded.


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


## 4. Text Splitting and FAISS Vector Database

Since LLMs have a strict context window limit, we cannot process the entire transcript at once.
This function divides the transcript into smaller, overlapping chunks (e.g., 1000 characters). It then uses HuggingFace embeddings (`BAAI/bge-small-en-v1.5`) to store these chunks in a FAISS vector database, allowing the Q&A chatbot to retrieve relevant information efficiently.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

def create_vector_db(transcript_text):
    print("1. Splitting text into overlapping chunks...")
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    text_chunks = text_splitter.split_text(transcript_text)

    print("2. Generating embeddings and building FAISS database...")
    embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
    vector_db = FAISS.from_texts(text_chunks, embeddings)

    return vector_db

print("[Success] FAISS Vector Database logic loaded.")

[Success] FAISS Vector Database logic loaded.


## 5. Llama 3.1 LLM & Smart Summarization (Map-Reduce)

This is the core reasoning engine of the system. We initialize the Llama 3.1-8b model here.
If the video is very long, this script automatically shifts to a **Map-Reduce Strategy**:
* A separate summary is created for each small chunk.
* Smart auto-retry and wait-time logic is implemented to handle API rate limits.
* Finally, all partial summaries are combined to generate a detailed, cohesive "Master Summary" in English.

In [5]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import time

llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.3)

# NAYA FUNCTION: Jo Rate Limits ko automatically handle karega
def safe_invoke(prompt_text, max_retries=3):
    for attempt in range(max_retries):
        try:
            return llm.invoke(prompt_text).content
        except Exception as e:
            error_msg = str(e)
            # Agar rate limit (429) hit ho jaye
            if "429" in error_msg or "rate_limit_exceeded" in error_msg:
                wait_time = 10 # 10 seconds ka safe buffer
                print(f"    [API Speed Limit Hit] Waiting {wait_time} seconds before retrying (Attempt {attempt + 1}/{max_retries})...")
                time.sleep(wait_time)
            else:
                # Agar koi aur serious error ho toh bahar nikal aayen
                raise e
    return "[Error: API Rate Limit exceeded persistently. Please try a shorter video or upgrade your Groq API tier.]"

def answer_video_question(vector_db, user_query):
    retriever = vector_db.as_retriever(search_kwargs={"k": 3})

    system_prompt = (
        "You are an intelligent AI assistant. "
        "Use the following pieces of retrieved context to answer the question. The context might be in Urdu, Hindi, or another language. "
        "If you don't know the answer, say that you don't know. "
        "CRITICAL INSTRUCTION: You MUST translate your understanding and always generate your final response completely in English."
        "\n\nContext:\n{context}"
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{input}"),
    ])

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    rag_chain = (
        {"context": retriever | format_docs, "input": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    return rag_chain.invoke(user_query)

def generate_video_summary(transcript_text):
    words = transcript_text.split()
    word_count = len(words)
    SAFE_CHUNK_SIZE = 1000

    if word_count <= SAFE_CHUNK_SIZE:
        print("Transcript is within safe limits. Generating direct summary...")

        if word_count < 50:
            length_instruction = "This is a very short clip. Provide a direct 1 to 2 sentence English translation/summary."
        elif word_count < 500:
            length_instruction = "Provide a concise summary using 3 to 5 bullet points in English."
        else:
            length_instruction = "Provide a detailed section-by-section summary in English."

        prompt = (
            "You are a professional AI analyzer. Read the following transcript. "
            f"{length_instruction} "
            "CRITICAL INSTRUCTION: The final output MUST be entirely in English.\n\n"
            f"Transcript:\n{transcript_text}"
        )
        try:
            return safe_invoke(prompt)
        except Exception as e:
            return f"Summary Error: {str(e)}"

    else:
        print(f"Warning: Transcript is {word_count} words long. Initiating Map-Reduce Strategy with Auto-Retry...")

        chunks = [" ".join(words[i:i + SAFE_CHUNK_SIZE]) for i in range(0, word_count, SAFE_CHUNK_SIZE)]
        partial_summaries = []

        for index, chunk in enumerate(chunks):
            print(f"Processing part {index + 1} of {len(chunks)}...")
            part_prompt = (
                "You are an AI summarizing a segment of a larger video transcript. "
                "Extract the core concepts and key points from this specific segment in English bullet points. "
                f"Segment:\n{chunk}"
            )
            try:
                # Use safe_invoke instead of direct invoke
                response_text = safe_invoke(part_prompt)
                partial_summaries.append(response_text)
                # Normal 5-second pause between chunks to respect steady limits
                time.sleep(5)
            except Exception as e:
                print(f"Error on part {index + 1}: {e}")
                partial_summaries.append(f"[Could not process part {index + 1}]")

        print("Merging all parts into the Final Master Summary...")
        combined_text = "\n\n--- NEXT PART ---\n\n".join(partial_summaries)

        final_prompt = (
            "You are an expert AI. I have provided summaries of different chronological parts of a long video below. "
            "Combine them into a single, cohesive, and highly comprehensive Final Master Summary in English. "
            "Organize the content logically using professional headings and detailed bullet points. "
            "Ensure the flow makes sense as a single document.\n\n"
            f"Combined Summaries:\n{combined_text}"
        )

        try:
            # Use safe_invoke for the final heavy merge
            final_response_text = safe_invoke(final_prompt)
            return final_response_text
        except Exception as e:
            return f"Error creating Final Master Summary: {str(e)}. \n\nSequential part-by-part summaries:\n\n{combined_text}"

print("[Success] Map-Reduce Summarizer with Smart Auto-Retry loaded.")

[Success] Map-Reduce Summarizer with Smart Auto-Retry loaded.


## 6. Gradio User Interface (Web App)

In this final section, we launch an interactive web application using Gradio. This interface contains two main tabs:
1. **Summary Tab:** Displays the detailed, bulleted summary of the processed video.
2. **Q&A Chatbot Tab:** Allows you to ask questions related to the video's content using RAG (Retrieval-Augmented Generation).

In [6]:
import gradio as gr

current_vector_db = None
current_transcript = ""

def handle_media(uploaded_file, drive_url):
    global current_vector_db, current_transcript

    if not uploaded_file and not drive_url:
        return "Error: No input provided.", "Please upload a file OR paste a Drive link."

    file_path = uploaded_file.name if uploaded_file else None

    # 1. Transcribe (Auto-translates to English via Whisper)
    transcript = process_media(file_path=file_path, drive_url=drive_url)
    if "Error" in transcript:
        return transcript, "Processing failed."

    current_transcript = transcript

    # 2. Build Vector DB (For Q&A Chatbot)
    current_vector_db = create_vector_db(transcript)

    # 3. Generate Dynamic Summary (Directly uses the full text for length analysis)
    summary = generate_video_summary(current_transcript)

    return "Media processed successfully! You can now ask questions in the Q&A tab.", summary

def chat_interface(user_message, chat_history):
    global current_vector_db

    if chat_history is None:
        chat_history = []

    if current_vector_db is None:
        chat_history.append({"role": "user", "content": user_message})
        chat_history.append({"role": "assistant", "content": "Please upload a media file or provide a Drive link first."})
        return "", chat_history

    ai_response = answer_video_question(current_vector_db, user_message)

    chat_history.append({"role": "user", "content": user_message})
    chat_history.append({"role": "assistant", "content": ai_response})

    return "", chat_history

# ----------------- UI DESIGN -----------------
with gr.Blocks() as demo:
    gr.Markdown("# Enterprise AI Audio/Video Analyzer")
    gr.Markdown("Upload a media file OR paste a public Google Drive link to automatically generate a dynamic length-aware summary and query the content.")

    # Yeh raha aapka Important Note jo interface par display hoga
    gr.Markdown("**Important Note:** For optimal transcription accuracy, please avoid using AI-generated voices. Original, human voices yield the best results.")

    with gr.Row():
        with gr.Column(scale=2):
            file_input = gr.File(label="Option 1: Upload Media (mp4, mp3, wav, etc.)")
            drive_input = gr.Textbox(label="Option 2: Google Drive Public Link", placeholder="https://drive.google.com/file/d/...")

        with gr.Column(scale=1):
            process_btn = gr.Button("Analyze Content", variant="primary", size="lg")
            status_output = gr.Textbox(label="System Status", interactive=False)

    with gr.Tabs():
        with gr.Tab("Summary"):
            summary_output = gr.Markdown("The dynamic document summary will appear here after processing...")

        with gr.Tab("Q&A Chatbot"):
            chatbot = gr.Chatbot(label="Content Assistant")
            msg_input = gr.Textbox(label="Type your question and press Enter...")
            clear_btn = gr.ClearButton([msg_input, chatbot])

    process_btn.click(
        fn=handle_media,
        inputs=[file_input, drive_input],
        outputs=[status_output, summary_output]
    )

    msg_input.submit(
        fn=chat_interface,
        inputs=[msg_input, chatbot],
        outputs=[msg_input, chatbot]
    )

print("Launching Gradio User Interface...")
demo.launch(debug=True, share=True)

Launching Gradio User Interface...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://44269dad4204873571.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Downloading...
From: https://drive.google.com/uc?id=1YZVkAAvHHbhby8QbzNx4ZaWHrSldvggn
To: /content/drive_media_1YZVkAAvHHbhby8QbzNx4ZaWHrSldvggn.mp4
100%|██████████| 55.4M/55.4M [00:00<00:00, 176MB/s]


Drive file downloaded. Starting processing...
Preparing media file for API limits (Chunking)...
File divided into 2 chunks. Transcribing via Groq API...
Processing chunk 1 of 2...
Processing chunk 2 of 2...
[Success] Full transcription and translation complete via Groq.
1. Splitting text into overlapping chunks...
2. Generating embeddings and building FAISS database...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Processing part 1 of 3...
Processing part 2 of 3...
Processing part 3 of 3...
Merging all parts into the Final Master Summary...
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://44269dad4204873571.gradio.live
